In [ ]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# 데이터 경로 설정 (노트북 위치 기준 상대경로)
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'data')
# 직접 경로 지정 (필요시)
DATA_DIR = r'c:\team-oldest-olist-analysis\data'

# 원본 데이터 로딩
orders       = pd.read_csv(os.path.join(DATA_DIR, 'olist_orders_dataset.csv'))
customers    = pd.read_csv(os.path.join(DATA_DIR, 'olist_customers_dataset.csv'))
order_items  = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_items_dataset.csv'))
payments     = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_payments_dataset.csv'))
reviews      = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_reviews_dataset.csv'))
products     = pd.read_csv(os.path.join(DATA_DIR, 'olist_products_dataset.csv'))
sellers      = pd.read_csv(os.path.join(DATA_DIR, 'olist_sellers_dataset.csv'))
category_tr  = pd.read_csv(os.path.join(DATA_DIR, 'product_category_name_translation.csv'))
geolocation  = pd.read_csv(os.path.join(DATA_DIR, 'olist_geolocation_dataset.csv'))

print("데이터 로딩 완료")
for name, df in [('orders', orders), ('customers', customers), ('order_items', order_items),
                 ('payments', payments), ('reviews', reviews), ('products', products),
                 ('sellers', sellers), ('category_tr', category_tr)]:
    print(f"  {name:15s}: {df.shape}")

## 1. 타임스탬프 변환

In [ ]:
# orders 타임스탬프 컬럼 datetime 변환
timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col])

# reviews 날짜 컬럼 변환
reviews['review_creation_date']    = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

# order_items shipping_limit_date 변환
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

print("타임스탬프 변환 완료")
orders[timestamp_cols].dtypes

## 2. 데이터 병합 (Master DataFrame 구성)

`customer_unique_id`를 기준으로 재구매 추적이 가능하도록 병합합니다.  
`customer_id`는 주문마다 새로 발급되므로 유저 단위 분석에는 `customer_unique_id`를 사용합니다.

In [ ]:
# 제품 카테고리 영문명 합치기
products = products.merge(category_tr, on='product_category_name', how='left')

# order_items에 상품·판매자 정보 추가
items_enriched = order_items.merge(products[['product_id', 'product_category_name', 'product_category_name_english']], 
                                    on='product_id', how='left')

# 주문별 결제 집계 (한 주문에 복수 결제수단 존재 가능)
payments_agg = (payments
    .groupby('order_id')
    .agg(
        total_payment_value=('payment_value', 'sum'),
        payment_type=('payment_type', lambda x: x.mode()[0]),  # 주 결제수단
        payment_installments=('payment_installments', 'max')
    )
    .reset_index()
)

# 주문별 아이템 집계 (다중 아이템 주문 처리)
items_agg = (items_enriched
    .groupby('order_id')
    .agg(
        item_count=('order_item_id', 'max'),
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        product_category=('product_category_name_english', lambda x: x.mode()[0] if x.notna().any() else np.nan)
    )
    .reset_index()
)

# 리뷰: order_id별 최신 1건 (중복 리뷰 제거)
reviews_dedup = (reviews
    .sort_values('review_answer_timestamp', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id', 'review_score', 'review_creation_date']]
)

# 마스터 병합
df = (orders
    .merge(customers[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id', how='left')
    .merge(payments_agg,   on='order_id', how='left')
    .merge(items_agg,      on='order_id', how='left')
    .merge(reviews_dedup,  on='order_id', how='left')
)

print(f"마스터 DataFrame shape: {df.shape}")
df.head(3)

## 3. 파생 변수 생성 (Feature Engineering)

In [ ]:
# --- 배송 관련 파생 변수 ---

# 배송 지연 여부 (실제 배송일 > 예상 배송일)
df['is_delayed'] = (
    df['order_delivered_customer_date'] > df['order_estimated_delivery_date']
).astype('Int8')  # nullable integer (NaN 허용)

# 지연 일수 (양수 = 지연, 음수 = 조기 배송)
df['delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

# --- 퍼널 구간별 소요 시간 (일 단위) ---

# 구간 1: 주문 → 결제 승인
df['time_purchase_to_approved'] = (
    df['order_approved_at'] - df['order_purchase_timestamp']
).dt.total_seconds() / 3600  # 시간 단위

# 구간 2: 결제 승인 → 물류사 인도 (판매자 처리 시간)
df['time_approved_to_carrier'] = (
    df['order_delivered_carrier_date'] - df['order_approved_at']
).dt.days

# 구간 3: 물류사 인도 → 고객 수령 (배송 소요 시간)
df['time_carrier_to_customer'] = (
    df['order_delivered_customer_date'] - df['order_delivered_carrier_date']
).dt.days

# 구간 4: 주문 → 고객 수령 (전체 리드타임)
df['total_lead_time_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

# --- 주문 시간 특성 ---
df['purchase_year']    = df['order_purchase_timestamp'].dt.year
df['purchase_month']   = df['order_purchase_timestamp'].dt.month
df['purchase_yearmonth'] = df['order_purchase_timestamp'].dt.to_period('M')
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek  # 0=월요일
df['purchase_hour']    = df['order_purchase_timestamp'].dt.hour

print("파생 변수 생성 완료")
derived_cols = ['is_delayed', 'delay_days', 'time_purchase_to_approved',
                'time_approved_to_carrier', 'time_carrier_to_customer', 'total_lead_time_days']
df[derived_cols].describe()

## 4. 결측치 처리 및 이상값 필터링

In [ ]:
# 전체 결측치 현황
print("=== 전체 결측치 현황 ===")
null_summary = df.isnull().sum()
print(null_summary[null_summary > 0].to_string())
print(f"\n전체 행 수: {len(df):,}")

# 주문 상태 분포 확인
print("\n=== order_status 분포 ===")
print(df['order_status'].value_counts())

In [ ]:
# 분석용 데이터셋: 'delivered' 상태만 유지 (배송 완료된 주문만)
df_delivered = df[df['order_status'] == 'delivered'].copy()
print(f"배송 완료 주문 수: {len(df_delivered):,} ({len(df_delivered)/len(df)*100:.1f}%)")

# 이상값 제거: 음수 리드타임 (데이터 오류)
negative_mask = (
    (df_delivered['total_lead_time_days'] < 0) |
    (df_delivered['time_approved_to_carrier'] < 0) |
    (df_delivered['time_carrier_to_customer'] < 0)
)
print(f"음수 리드타임 이상값: {negative_mask.sum()}건 제거")
df_delivered = df_delivered[~negative_mask]

# review_score 결측 → 리뷰 미작성 (0으로 채우지 않고 NaN 유지, 분석 시 제외)
print(f"\nreview_score 결측 (리뷰 미작성): {df_delivered['review_score'].isna().sum():,}건")

print(f"\n최종 분석용 DataFrame shape: {df_delivered.shape}")

## 5. 전처리 결과 저장

In [ ]:
# period 타입은 CSV 저장 불가 → 문자열로 변환
df_delivered['purchase_yearmonth'] = df_delivered['purchase_yearmonth'].astype(str)

# 전체 주문 포함 버전 (퍼널 분석용)
df_all_save = df.copy()
df_all_save['purchase_yearmonth'] = df_all_save['purchase_yearmonth'].astype(str)

SAVE_DIR = r'c:\team-oldest-olist-analysis\data'
df_delivered.to_csv(os.path.join(SAVE_DIR, 'olist_master_delivered.csv'), index=False)
df_all_save.to_csv(os.path.join(SAVE_DIR,  'olist_master_all.csv'),       index=False)

print("저장 완료")
print(f"  olist_master_delivered.csv : {df_delivered.shape}")
print(f"  olist_master_all.csv       : {df_all_save.shape}")
print("\n컬럼 목록:")
print(df_delivered.columns.tolist())